# Week 3 - Exercises

## Exercise 1: Collecting Research Articles from IC2S2 Authors

In [2]:
import pandas as pd

ic2s2_authors = pd.read_csv("week2_openalex_authors.csv")
print(f"total authors: {len(ic2s2_authors)}")
print(ic2s2_authors['works_count'].describe())

total authors: 1297
count    1297.000000
mean      111.066307
std       199.614896
min         1.000000
25%        13.000000
50%        46.000000
75%       127.000000
max      2790.000000
Name: works_count, dtype: float64


In [3]:
# filter authors: works_count between 5 and 5000
ic2s2_authors = ic2s2_authors[(ic2s2_authors['works_count'] >= 5) & (ic2s2_authors['works_count'] <= 5000)]
print(f"authors after filtering: {len(ic2s2_authors)}")

authors after filtering: 1141


In [ ]:
import requests

api_key = "zhj01t4KDUqWAHNFrcZrok"

# test with one author
sample_id = ic2s2_authors['openalex_id'].iloc[1]  # Aaron Nichols
print(f"testing with: {sample_id}")

response = requests.get(
    "https://api.openalex.org/works",
    params={
        "api_key": api_key,
        "filter": f"author.id:{sample_id}",
        "per_page": 1
    }
)

data = response.json()
print(f"total works for this author: {data['meta']['count']}")
print(f"cost of this call: ${data['meta']['cost_usd']}")

testing with: https://openalex.org/A5089395967
total works for this author: 10
cost of this call: $0.0001


In [5]:
# inspect the structure of one work
work = data['results'][0]
print(work.keys())

dict_keys(['id', 'doi', 'title', 'display_name', 'publication_year', 'publication_date', 'ids', 'language', 'primary_location', 'type', 'indexed_in', 'open_access', 'authorships', 'institutions', 'countries_distinct_count', 'institutions_distinct_count', 'corresponding_author_ids', 'corresponding_institution_ids', 'apc_list', 'apc_paid', 'fwci', 'has_fulltext', 'cited_by_count', 'citation_normalized_percentile', 'cited_by_percentile_year', 'biblio', 'is_retracted', 'is_paratext', 'is_xpac', 'primary_topic', 'topics', 'keywords', 'concepts', 'mesh', 'locations_count', 'locations', 'best_oa_location', 'sustainable_development_goals', 'awards', 'funders', 'has_content', 'content_urls', 'referenced_works_count', 'referenced_works', 'related_works', 'abstract_inverted_index', 'counts_by_year', 'updated_date', 'created_date'])


In [6]:
# inspect the fields we need to extract
print("id:", work['id'])
print("publication_year:", work['publication_year'])
print("cited_by_count:", work['cited_by_count'])
print("title:", work['title'])
print()
# authorships is a list — let's see what one entry looks like
print("authorships (first entry):", work['authorships'][0])
print()
# abstract is an inverted index — let's peek at it
print("abstract_inverted_index (first 3 items):", dict(list(work['abstract_inverted_index'].items())[:3]) if work['abstract_inverted_index'] else "None")

id: https://openalex.org/W2408628302
publication_year: 2016
cited_by_count: 61
title: Music As a Sacred Cue? Effects of Religious Music on Moral Behavior

authorships (first entry): {'author_position': 'first', 'author': {'id': 'https://openalex.org/A5048616798', 'display_name': 'Martin Lang', 'orcid': 'https://orcid.org/0000-0002-2231-1059'}, 'institutions': [{'id': 'https://openalex.org/I21449261', 'display_name': 'Masaryk University', 'ror': 'https://ror.org/02j46qs45', 'country_code': 'CZ', 'type': 'education', 'lineage': ['https://openalex.org/I21449261']}, {'id': 'https://openalex.org/I140172145', 'display_name': 'University of Connecticut', 'ror': 'https://ror.org/02der9h97', 'country_code': 'US', 'type': 'education', 'lineage': ['https://openalex.org/I140172145']}], 'countries': ['CZ', 'US'], 'is_corresponding': True, 'raw_author_name': 'Martin Lang', 'raw_affiliation_strings': ['Department of Anthropology, University of ConnecticutStorrs, CT, USA; LEVYNA Laboratory for the Exp

In [ ]:
# extract author ids from the authorships list
author_ids = [a['author']['id'] for a in work['authorships']]
print(f"number of authors: {len(author_ids)}")
print(author_ids)

number of authors: 6
['https://openalex.org/A5048616798', 'https://openalex.org/A5018921866', 'https://openalex.org/A5034964074', 'https://openalex.org/A5089395967', 'https://openalex.org/A5109388873', 'https://openalex.org/A5016308056']


In [8]:
# test with citation filter added
response = requests.get(
    "https://api.openalex.org/works",
    params={
        "api_key": api_key,
        "filter": f"author.id:{sample_id},cited_by_count:>10",
        "per_page": 1
    }
)

data = response.json()
print(f"works with >10 citations: {data['meta']['count']} (out of 10 total)")

works with >10 citations: 2 (out of 10 total)


In [9]:
# let's look up the concept IDs we need
# we can use the OpenAlex concepts endpoint
concepts_to_find = ["Sociology", "Psychology", "Economics", "Political Science", 
                     "Mathematics", "Physics", "Computer Science"]

for c in concepts_to_find:
    r = requests.get(
        "https://api.openalex.org/concepts",
        params={"api_key": api_key, "filter": f"display_name.search:{c},level:0"}
    )
    results = r.json()['results']
    if results:
        print(f"{c}: {results[0]['id']}")
    else:
        print(f"{c}: NOT FOUND")

Sociology: https://openalex.org/C144024400
Psychology: https://openalex.org/C15744967
Economics: https://openalex.org/C162324750
Political Science: https://openalex.org/C17744445
Mathematics: https://openalex.org/C33923547
Physics: https://openalex.org/C121332964
Computer Science: https://openalex.org/C41008148


In [10]:
# define concept ID groups (just the short IDs work too)
social = "C144024400|C15744967|C162324750|C17744445"
quant = "C33923547|C121332964|C41008148"

# test the full filter on our sample author
response = requests.get(
    "https://api.openalex.org/works",
    params={
        "api_key": api_key,
        "filter": f"author.id:{sample_id},cited_by_count:>10,concepts.id:{social},concepts.id:{quant}",
        "per_page": 1
    }
)

data = response.json()
print(f"works matching all filters: {data['meta']['count']} (out of 10 total)")

works matching all filters: 0 (out of 10 total)


In [11]:
# test with a different author — pick someone more likely to be in CSS
# let's find someone with a high h_index, more likely to be a known CSS researcher
top_authors = ic2s2_authors.nlargest(5, 'h_index')[['display_name', 'openalex_id', 'h_index']]
print(top_authors)

      display_name                       openalex_id  h_index
570  Jian‐Kang Zhu  https://openalex.org/A5049293054      190
823   Michaël Maes  https://openalex.org/A5032880777      141
995    Robert West  https://openalex.org/A5059645286      132
498   Chunhua Shen  https://openalex.org/A5006294869      122
32   Alex Pentland  https://openalex.org/A5007176508      121


In [12]:
# test with Alex Pentland
pentland_id = "https://openalex.org/A5007176508"

response = requests.get(
    "https://api.openalex.org/works",
    params={
        "api_key": api_key,
        "filter": f"author.id:{pentland_id},cited_by_count:>10,concepts.id:{social},concepts.id:{quant}",
        "per_page": 1
    }
)

data = response.json()
print(f"Pentland works matching all filters: {data['meta']['count']}")

Pentland works matching all filters: 252


In [13]:
# let's estimate how many API calls we'll need
# first, check total works across ALL authors with our filters
# we can batch author IDs with | (OR) — let's test with a small batch

batch = ic2s2_authors['openalex_id'].iloc[:25]
author_filter = "|".join(batch.values)

response = requests.get(
    "https://api.openalex.org/works",
    params={
        "api_key": api_key,
        "filter": f"author.id:{author_filter},cited_by_count:>10,concepts.id:{social},concepts.id:{quant}",
        "per_page": 1
    }
)

data = response.json()
print(f"works from first 25 authors: {data['meta']['count']}")
print(f"cost of this call: ${data['meta']['cost_usd']}")

works from first 25 authors: 290
cost of this call: $0.0001


In [14]:
# rough estimate:
# 1141 authors / 25 per batch = ~46 batches
# each batch returns ~290 works on average (probably varies a lot)
# but let's get a better estimate — check total works for all authors

import math

n_batches = math.ceil(len(ic2s2_authors) / 25)
print(f"number of batches: {n_batches}")

# estimated total works (rough): 290 works per 25 authors
est_total_works = (len(ic2s2_authors) / 25) * 290
est_pages = est_total_works / 100  # 100 results per page
est_calls = n_batches + est_pages  # batch calls + pagination calls
est_cost = est_calls * 0.0001

print(f"estimated total works: {est_total_works:.0f}")
print(f"estimated total API calls: {est_calls:.0f}")
print(f"estimated cost: ${est_cost:.2f}")

number of batches: 46
estimated total works: 13236
estimated total API calls: 178
estimated cost: $0.02


In [15]:
# let's first write a function to fetch all works for a batch of author IDs
# using cursor paging as recommended by the exercise

def fetch_works_for_batch(author_ids, api_key, social, quant):
    """Fetch all works for a batch of authors, handling pagination with cursor."""
    all_works = []
    author_filter = "|".join(author_ids)
    cursor = "*"  # initial cursor value
    
    while cursor:
        response = requests.get(
            "https://api.openalex.org/works",
            params={
                "api_key": api_key,
                "filter": f"author.id:{author_filter},cited_by_count:>10,concepts.id:{social},concepts.id:{quant}",
                "per_page": 100,
                "cursor": cursor
            }
        )
        data = response.json()
        results = data.get('results', [])
        all_works.extend(results)
        
        # get next cursor — None means we've reached the end
        cursor = data['meta'].get('next_cursor')
    
    return all_works

# test it on our first batch of 25
batch = ic2s2_authors['openalex_id'].iloc[:25].values
works = fetch_works_for_batch(batch, api_key, social, quant)
print(f"fetched {len(works)} works from first batch")

fetched 290 works from first batch


In [20]:
from tqdm import tqdm
from joblib import Parallel, delayed

# split all author IDs into batches of 25
all_ids = ic2s2_authors['openalex_id'].values
batches = [all_ids[i:i+25] for i in range(0, len(all_ids), 25)]
print(f"total batches: {len(batches)}")

# fetch all works in parallel using 5 cores
all_works_nested = Parallel(n_jobs=5)(
    delayed(fetch_works_for_batch)(batch, api_key, social, quant)
    for batch in tqdm(batches)
)

# flatten the list of lists into one list
all_works = [w for batch_works in all_works_nested for w in batch_works]
print(f"total works after proper flattening: {len(all_works)}")

# check uniqueness
unique_ids = set(w['id'] for w in all_works)
print(f"unique works: {len(unique_ids)}")

total batches: 46


100%|██████████| 46/46 [00:59<00:00,  1.30s/it]


total works after proper flattening: 15657
unique works: 13884


In [21]:
# extract the fields we need, filtering out works with 10+ authors
papers = []
abstracts = []

for w in all_works:
    author_ids = [a['author']['id'] for a in w['authorships']]
    
    # skip works with 10 or more authors
    if len(author_ids) >= 10:
        continue
    
    papers.append({
        'id': w['id'],
        'publication_year': w['publication_year'],
        'cited_by_count': w['cited_by_count'],
        'author_ids': author_ids
    })
    
    abstracts.append({
        'id': w['id'],
        'title': w['title'],
        'abstract_inverted_index': w['abstract_inverted_index']
    })

print(f"works after filtering <10 authors: {len(papers)} (removed {len(all_works) - len(papers)})")

works after filtering <10 authors: 14489 (removed 1168)


In [22]:
df_papers = pd.DataFrame(papers)
df_abstracts = pd.DataFrame(abstracts)

print(f"D2 - IC2S2 papers: {df_papers.shape}")
print(f"D3 - IC2S2 abstracts: {df_abstracts.shape}")
print()
print(df_papers.head())

D2 - IC2S2 papers: (14489, 4)
D3 - IC2S2 abstracts: (14489, 3)

                                 id  publication_year  cited_by_count  \
0  https://openalex.org/W2047940964            2004.0            7334   
1  https://openalex.org/W3103362336            2018.0            6724   
2  https://openalex.org/W2157082398            2008.0            2163   
3  https://openalex.org/W2974087526            2019.0            1732   
4  https://openalex.org/W3133423348            2021.0            1712   

                                          author_ids  
0  [https://openalex.org/A5014647140, https://ope...  
1  [https://openalex.org/A5014647140, https://ope...  
2  [https://openalex.org/A5014647140, https://ope...  
3  [https://openalex.org/A5054642819, https://ope...  
4  [https://openalex.org/A5076143079, https://ope...  


In [23]:
# drop duplicate works
df_papers = df_papers.drop_duplicates(subset='id')
df_abstracts = df_abstracts.drop_duplicates(subset='id')

print(f"D2 - IC2S2 papers after dedup: {df_papers.shape}")
print(f"D3 - IC2S2 abstracts after dedup: {df_abstracts.shape}")

D2 - IC2S2 papers after dedup: (12991, 4)
D3 - IC2S2 abstracts after dedup: (12991, 3)


In [24]:
df_papers.to_json("week3_ic2s2_papers.json", orient='records')
df_abstracts.to_json("week3_ic2s2_abstracts.json", orient='records')

print("D2 saved to week3_ic2s2_papers.json")
print("D3 saved to week3_ic2s2_abstracts.json")

D2 saved to week3_ic2s2_papers.json
D3 saved to week3_ic2s2_abstracts.json


In [25]:
# count unique researchers across all papers
all_author_ids = set()
for ids in df_papers['author_ids']:
    all_author_ids.update(ids)

print(f"unique works: {len(df_papers)}")
print(f"unique researchers who co-authored these works: {len(all_author_ids)}")

unique works: 12991
unique researchers who co-authored these works: 20001


### Exercise 1: Reflection Questions

**Dataset summary.**
the ic2s2 papers dataset (D2) contains 12,991 unique works. across these works, there are 20,001 unique researchers who have co-authored at least one paper.

**Efficiency in code.**
we used several strategies to speed up data collection. first, we batched 25 author IDs per API request using the OR filter syntax, reducing ~1,141 individual calls to just 46. second, we applied filters directly in the API call (cited_by_count >10, concept filters for social science + quantitative fields) so only relevant works were returned. third, we set per_page=100 to minimize pagination calls. finally, we used joblib's Parallel with 5 cores to run batches simultaneously. combined, these brought total fetch time down to about 1 minute.

**Filtering criteria and dataset relevance.**
the filters help keep the dataset focused on established, interdisciplinary css research, but they introduce biases. the >10 citations threshold excludes recent work that hasn't had time to accumulate citations, underrepresenting emerging topics and early-career researchers. the concept filter requiring both a social science and a quantitative field excludes qualitative css work like computational ethnography. the <10 authors filter removes large collaborative projects, which are increasingly common. the works_count 5–5000 filter removes very inactive researchers but also potentially excludes newcomers. overall, the dataset skews toward well-established, quantitative, small-team research.

## Exercise 2:

## Exercise 3: